# Filtered aperture E/B shear maps on HEALPix (DP2, tract 9813, shapeHSM)

Alex Broughton · April 2026. Rubin stack and Butler access required.


## Introduction

This notebook builds Schirmer-filtered aperture **E- and B-mode** shear maps on HEALPix for DP2 tract 9813 HSM (shapeHSM Regauss) shear (weighted aperture statistics, not Kaiser–Squires).

- **E-mode:** tangential shear \(g_t\) with \(Q\); **B-mode:** cross shear \(g_\times\) with the same \(Q\) (near zero for pure lensing).
- **Geometry:** each HEALPix RING cell is evaluated at its center; the KD-tree search radius is a configurable multiple of \(R_s\) (not \(R_s\) itself).
- **Weights:** inverse variance from catalog shear covariances.


## 1.0 Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

from lsst.daf.butler import Butler

CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",  # Butler repository root
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",  # Input collection
    "SKYMAP": "lsst_cells_v2",  # Skymap name for Butler
    "INSTRUMENT": "LSSTCam",  # Instrument dimension
    "TRACT": 9813,  # Tract id
    # "TRACT": 3725,  # Tract id
    "RS_INPUT_PIX": 10000,  # Schirmer scale in native catalog pixels
    "PIX_SCALE_ARCSEC": 0.2,  # Arcsec per pixel (converts RS_INPUT_PIX to sky angle)
    "SEARCH_RADIUS_RS_MULT": 3.0,  # cKDTree search radius = this × Rs (in arcmin)
    "NSIDE": 2048,  # HEALPix NSIDE (RING); use a power of 2
    "PEAK_SN_MIN": 3.0,  # Peak S/N cut; use None to disable
}

CONFIG["RS_ARCSEC"] = float(CONFIG["RS_INPUT_PIX"]) * float(CONFIG["PIX_SCALE_ARCSEC"])
CONFIG["RS_ARCMIN"] = CONFIG["RS_ARCSEC"] / 60.0
CONFIG["RS_DEG"] = CONFIG["RS_ARCSEC"] / 3600.0
CONFIG["THETA_MAX_ARCMIN"] = float(CONFIG["SEARCH_RADIUS_RS_MULT"]) * CONFIG["RS_ARCMIN"]
CONFIG["NSIDE"] = int(CONFIG["NSIDE"])
_mean_arcsec = float(np.rad2deg((4.0 * np.pi / 12.0) ** 0.5 / CONFIG["NSIDE"]) * 3600.0)

%matplotlib inline
plt.rcParams.update({"font.size": 12})

butler = Butler(
    CONFIG["REPO"],
    collections=[CONFIG["COLLECTION"]],
    skymap=CONFIG["SKYMAP"],
    instrument=CONFIG["INSTRUMENT"],
)

_npix = hp.nside2npix(CONFIG["NSIDE"])
print("TRACT =", CONFIG["TRACT"])
print(
    "Schirmer Rs =",
    CONFIG["RS_INPUT_PIX"],
    "pix @",
    CONFIG["PIX_SCALE_ARCSEC"],
    "arcsec/pix →",
    f"{CONFIG['RS_ARCMIN']:.3f} arcmin",
)
print(
    "search radius THETA_MAX_ARCMIN =",
    f"{CONFIG['THETA_MAX_ARCMIN']:.3f}",
    f"({CONFIG['SEARCH_RADIUS_RS_MULT']}×Rs) | NSIDE =",
    CONFIG["NSIDE"],
    f"(npix={_npix}; ~mean spacing {_mean_arcsec:.1f} arcsec)",
)


In [ ]:
def schirmer_weight(r, rs):
    """Schirmer filter Q(r/Rs); r and rs in the same angular units (degrees here)."""
    r = np.asarray(r, dtype=np.float64)
    rs = float(rs)
    x = r / rs
    a, b, c, d, xc = 6.0, 150.0, 47.0, 50.0, 0.15
    q = 1.0 / (1.0 + np.exp(a - b * x) + np.exp(d * x - c))
    ratio = x / xc
    with np.errstate(divide="ignore", invalid="ignore"):
        inner = np.tanh(ratio) / ratio
    inner = np.where(np.abs(ratio) < 1e-14, 1.0, inner)
    q = q * inner
    q = np.where(np.isfinite(q), q, 0.0)
    return q


def gamma_tx_sky(ra, dec, g1, g2, center):
    """Tangential g_t and cross g_x shear about center (degrees, small-angle plane)."""
    ra = np.asarray(ra, dtype=np.float64)
    dec = np.asarray(dec, dtype=np.float64)
    g1 = np.asarray(g1, dtype=np.float64)
    g2 = np.asarray(g2, dtype=np.float64)
    ra0, dec0 = center
    cos_dec0 = np.cos(np.deg2rad(dec0))
    dx = (ra - ra0) * cos_dec0
    dy = dec - dec0
    phi = np.arctan2(dy, dx)
    c2 = np.cos(2.0 * phi)
    s2 = np.sin(2.0 * phi)
    gt = -(g1 * c2 + g2 * s2)
    gx = -g1 * s2 + g2 * c2
    return gt, gx


def aperture_mass_and_sn_at_center(
    ra_c,
    dec_c,
    ra_all,
    dec_all,
    g1_all,
    g2_all,
    *,
    theta_max_arcmin,
    rs_deg,
    randomize=False,
    w_all=None,
):
    """Filtered aperture E- and B-mode statistics and S/N at one center; optional w_all weights galaxies.

    Returns (ap_e, sn_e, ap_b, sn_b) using Schirmer Q with tangential gt (E-type) and cross gx (B-type).
    """
    theta_max_deg = float(theta_max_arcmin) / 60.0
    dra = ra_all - ra_c
    ddec = dec_all - dec_c
    # Same tangent-plane metric as gamma_tx_sky (dx = dra * cos dec0) so Q(r) matches angular r.
    cos_dec0 = np.cos(np.deg2rad(dec_c))
    dtheta = np.hypot(dra * cos_dec0, ddec)
    m = dtheta <= theta_max_deg
    if not np.any(m):
        return np.nan, np.nan, np.nan, np.nan

    ra_s = ra_all[m]
    dec_s = dec_all[m]
    g1_s = g1_all[m]
    g2_s = g2_all[m]
    d_theta = dtheta[m]
    n_g = int(ra_s.size)

    if w_all is None:
        w_s = np.ones(n_g, dtype=np.float64)
    else:
        w_s = np.asarray(w_all, dtype=np.float64)[m]

    gt, gx = gamma_tx_sky(ra_s, dec_s, g1_s, g2_s, center=(ra_c, dec_c))

    if randomize:
        rng = np.random.default_rng()
        ang = rng.uniform(0.0, 2.0 * np.pi)
        gt = gt * (-np.cos(ang))

    Qi = schirmer_weight(d_theta, rs_deg)
    w_sum = float(np.sum(w_s))
    if w_sum <= 0.0 or not np.isfinite(w_sum):
        return np.nan, np.nan, np.nan, np.nan

    ap_e = np.sum(Qi * gt * w_s) / w_sum
    ap_b = np.sum(Qi * gx * w_s) / w_sum
    denom = np.sqrt(np.sum((Qi**2) * (g1_s**2 + g2_s**2) * (w_s**2)))
    if denom <= 0.0 or not np.isfinite(denom):
        return ap_e, np.nan, ap_b, np.nan
    sn_e = np.sqrt(2.0) * (w_sum * ap_e) / denom
    sn_b = np.sqrt(2.0) * (w_sum * ap_b) / denom
    return ap_e, sn_e, ap_b, sn_b


def find_peaks_shear_healpix(
    ipix_eval,
    ap_mass,
    counts,
    sn_ratio,
    nside,
    theta_max_arcmin,
    sn_min=None,
):
    """Strict local maxima on HEALPix neighbors; density cut counts/(πθ²)>0.5 arcmin⁻²; optional sn_min."""
    ipix_eval = np.asarray(ipix_eval, dtype=np.int64)
    ap_mass = np.asarray(ap_mass, dtype=np.float64)
    counts = np.asarray(counts, dtype=np.float64)
    sn_ratio = np.asarray(sn_ratio, dtype=np.float64)
    idx_of = {int(ip): k for k, ip in enumerate(ipix_eval)}
    theta_max_arcmin = float(theta_max_arcmin)
    peaks_ipix = []

    for k, ip in enumerate(ipix_eval):
        ap = ap_mass[k]
        if not np.isfinite(ap):
            continue
        nbrs = hp.get_all_neighbours(nside, int(ip), nest=False)
        is_peak = True
        for nj in nbrs:
            if nj < 0:
                continue
            j = idx_of.get(int(nj))
            if j is None:
                continue
            if ap <= ap_mass[j]:
                is_peak = False
                break
        if not is_peak:
            continue
        density = counts[k] / (np.pi * (theta_max_arcmin**2))
        if density <= 0.5:
            continue
        if sn_min is not None:
            if not (np.isfinite(sn_ratio[k]) and sn_ratio[k] >= sn_min):
                continue
        peaks_ipix.append(int(ip))

    return np.asarray(peaks_ipix, dtype=np.int64)



## 2.0 Load data

Fetch the tract-level shear table from the Butler (one Astropy table for the tract).


In [ ]:
# data = butler.get("object_shear_all", dataId={"tract": CONFIG["TRACT"]})
data = butler.get("object", dataId={"tract": CONFIG["TRACT"]})
print(f"Rows: {len(data):,}")


## 3.0 Source selection



In [ ]:
mask = (data['i_i_flag'] == False)

mask &= (data['i_iPSF_flag'] == False)
mask &= (data['i_hsmShapeRegauss_flag'] == False)
mask &= (data['i_gaapFlux_flag'] == False)
mask &= (data['sersic_no_data_flag'] == False)
mask &= (data['sersic_unknown_flag'] == False)
mask &= (data['shape_flag'] == False)
mask &= (data['i_hsmShapeRegauss_e1']**2 + data['i_hsmShapeRegauss_e2']**2) <= 4
# mask &= (data['i_gaapFlux_flag'] == False)
# mask &= (data['i_gaapFlux_flag'] == False)

reso_factor = 1 - (data['i_ixxPSF'] + data['i_iyyPSF'])/(data['i_ixx'] + data['i_iyy'])
mask &= reso_factor > 0.2
# mask &= (data['gauss_g1']**2 + data['gauss_g2']**2) <= 0.4**2

# Mag cut?

shear_catalog = data[mask]
print("Sources before quality cuts:", len(data))
print("Sources after quality cuts:", len(shear_catalog))


## 4.0 Map on HEALPix

Unique HEALPix pixels that contain at least one selected galaxy are evaluated at pixel centers. For each center, gather galaxies within the search radius, apply the Schirmer weight to tangential shear \(g_t\) (E-type aperture statistic, `ap_mass_map` / `sn_ratio_map`) and to cross shear \(g_\times\) (B-type, `ap_bmode_map` / `sn_bmode_map`), and fill `ipix_eval`, `counts_map`, and those four map arrays.


In [ ]:
def _col(tbl, name):
    return np.array(tbl[name]).astype(np.float64, copy=False)

# A360 Calibration
R, m_mean = 0.8511839657185559, -0.13584282915829576

ra_deg = _col(shear_catalog, "coord_ra")
dec_deg = _col(shear_catalog, "coord_dec")
# g1 = _col(shear_catalog, "i_hsmShapeRegauss_e1")
# g2 = _col(shear_catalog, "i_hsmShapeRegauss_e2")
e1_0 = _col(shear_catalog, "i_hsmShapeRegauss_e1")
e2_0 = _col(shear_catalog, "i_hsmShapeRegauss_e2")
# e1_0 -= np.mean(e1_0)
# e2_0 -= np.mean(e2_0)
g1 = (e1_0 / (2.0 * R)) / (1.0 + m_mean)
g2 = (e2_0 / (2.0 * R)) / (1.0 + m_mean)


# Weights: 1 / (Var(g1) + Var(g2)) from catalog covariances.
# _var_sum = _col(shear_catalog, "gauss_g1_g1_Cov") + _col(shear_catalog, "gauss_g2_g2_Cov")
_var_sum = (0.26**2 + _col(shear_catalog, 'i_hsmShapeRegauss_sigma'))
# _var_sum = np.ones(len(shear_catalog))
with np.errstate(divide="ignore", invalid="ignore"):
    w_gal = np.where(np.isfinite(_var_sum) & (_var_sum > 0.0), 1.0 / _var_sum, 0.0)

print("N galaxies:", ra_deg.size)
print("Galaxies with positive finite weight:", int(np.sum(w_gal > 0)))


In [ ]:
theta_max_arcmin = CONFIG["THETA_MAX_ARCMIN"]
rs_deg = float(CONFIG["RS_DEG"])
nside = int(CONFIG["NSIDE"])
theta_max_deg = theta_max_arcmin / 60.0

ipix_gal = hp.ang2pix(nside, ra_deg, dec_deg, nest=False, lonlat=True)
ipix_eval = np.unique(ipix_gal)

xy = np.column_stack([ra_deg, dec_deg])
tree = cKDTree(xy)

n_cell = int(ipix_eval.size)
ap_mass_map = np.full(n_cell, np.nan, dtype=np.float64)
sn_ratio_map = np.full(n_cell, np.nan, dtype=np.float64)
ap_bmode_map = np.full(n_cell, np.nan, dtype=np.float64)
sn_bmode_map = np.full(n_cell, np.nan, dtype=np.float64)
counts_map = np.zeros(n_cell, dtype=np.int32)

for k, ip in enumerate(tqdm(ipix_eval, desc="HEALPix cells")):
    ra_c, dec_c = hp.pix2ang(nside, int(ip), nest=False, lonlat=True)
    idx = tree.query_ball_point([float(ra_c), float(dec_c)], r=theta_max_deg)
    if not idx:
        continue
    idx = np.asarray(idx, dtype=np.int64)
    ra_s = ra_deg[idx]
    dec_s = dec_deg[idx]
    g1_s = g1[idx]
    g2_s = g2[idx]
    counts_map[k] = int(ra_s.size)
    ap_mass_map[k], sn_ratio_map[k], ap_bmode_map[k], sn_bmode_map[k] = aperture_mass_and_sn_at_center(
        float(ra_c),
        float(dec_c),
        ra_s,
        dec_s,
        g1_s,
        g2_s,
        theta_max_arcmin=theta_max_arcmin,
        rs_deg=rs_deg,
        randomize=False,
        w_all=w_gal[idx],
    )

print("N HEALPix cells evaluated:", n_cell)
print("Finite E-mode (aperture mass) cells:", int(np.isfinite(ap_mass_map).sum()))
print("Finite B-mode cells:", int(np.isfinite(ap_bmode_map).sum()))


### Save Tables

In [ ]:
from astropy.table import Table

In [ ]:
ra_pix, dec_pix = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
ra_pix = np.asarray(ra_pix, dtype=np.float64)
dec_pix = np.asarray(dec_pix, dtype=np.float64)


In [ ]:
aper = Table(data=[ra_pix, dec_pix, ap_mass_map, ap_bmode_map], names=["RA", "DEC", "E_ap", "B_ap"])

In [ ]:
aper.write(f"./data/{CONFIG['TRACT']}_shapeHSM_massmap.fits", format="fits", overwrite=True)

## 5.0 Peaks

Local maxima on the HEALPix neighbor graph, with the same surface-density cut as above and an optional S/N threshold from config.


In [ ]:
sn_min = CONFIG.get("PEAK_SN_MIN")
peak_ipix = find_peaks_shear_healpix(
    ipix_eval,
    ap_mass_map,
    counts_map,
    sn_ratio_map,
    nside,
    theta_max_arcmin=theta_max_arcmin,
    sn_min=sn_min,
)
print("N peaks (with current cuts):", int(peak_ipix.size))


## 6.0 Figures

RA/Dec scatter maps, S/N with peaks, and optional gnomonic healpy views when `NSIDE` keeps the full map in memory.


In [ ]:
ra_pix, dec_pix = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
ra_pix = np.asarray(ra_pix, dtype=np.float64)
dec_pix = np.asarray(dec_pix, dtype=np.float64)

msk = counts_map > 0
if msk.any():
    m_lo, m_hi = np.nanpercentile(ap_mass_map[msk], [5, 95])
    sn_lo, sn_hi = np.nanpercentile(sn_ratio_map[msk], [5, 95])
    b_lo, b_hi = np.nanpercentile(ap_bmode_map[msk], [5, 95])
    snb_lo, snb_hi = np.nanpercentile(sn_bmode_map[msk], [5, 95])
else:
    m_lo = m_hi = sn_lo = sn_hi = b_lo = b_hi = snb_lo = snb_hi = None

pt_size = max(1.0, min(8.0, 500_000.0 / max(int(ipix_eval.size), 1)))

MAX_DENSE_NPIX = 4_000_000
_npix_hp = hp.nside2npix(nside)
do_gnomonic = _npix_hp <= MAX_DENSE_NPIX
if do_gnomonic:
    rot_ra = float(np.median(ra_pix))
    rot_dec = float(np.median(dec_pix))
    sn_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    sn_dense[ipix_eval] = sn_ratio_map
    hp_sm = np.copy(sn_dense)
    hp_sm[~np.isfinite(hp_sm)] = hp.UNSEEN
    snb_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    snb_dense[ipix_eval] = sn_bmode_map
    hp_smb = np.copy(snb_dense)
    hp_smb[~np.isfinite(hp_smb)] = hp.UNSEEN
else:
    hp_sm = hp_smb = None


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
sc0 = axes[0].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=ap_mass_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=m_lo,
    vmax=m_hi,
    linewidths=0,
    alpha=0.85,
)
axes[0].set_xlabel("RA (deg)")
axes[0].set_ylabel("Dec (deg)")
axes[0].set_title(r"Aperture E-mode ($g_t \times Q$)")
axes[0].invert_xaxis()
fig.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.02)

sc1 = axes[1].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=sn_lo,
    vmax=sn_hi,
    linewidths=0,
    alpha=0.85,
)
axes[1].set_xlabel("RA (deg)")
axes[1].set_ylabel("Dec (deg)")
axes[1].set_title(r"E-mode $\mathcal{S}/\mathcal{N}$")
axes[1].invert_xaxis()
fig.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.02)

logc = np.log10(counts_map.astype(np.float64) + 0.1)
sc2 = axes[2].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=logc[msk],
    s=pt_size,
    cmap="magma",
    linewidths=0,
    alpha=0.85,
)
axes[2].set_xlabel("RA (deg)")
axes[2].set_ylabel("Dec (deg)")
axes[2].set_title(r"$\log_{10}$(counts in search disc) + 0.1")
axes[2].invert_xaxis()
fig.colorbar(sc2, ax=axes[2], fraction=0.046, pad=0.02)

fig.suptitle(
    f"Tract {CONFIG['TRACT']}, NSIDE={nside}, Rs={CONFIG['RS_ARCMIN']:.2f}', search={theta_max_arcmin:.1f}'",
    y=1.02,
)


In [ ]:
fig_eb, axes_eb = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
scb0 = axes_eb[0].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=ap_bmode_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=b_lo,
    vmax=b_hi,
    linewidths=0,
    alpha=0.85,
)
axes_eb[0].set_xlabel("RA (deg)")
axes_eb[0].set_ylabel("Dec (deg)")
axes_eb[0].set_title(r"Aperture B-mode ($g_\times \times Q$)")
axes_eb[0].invert_xaxis()
fig_eb.colorbar(scb0, ax=axes_eb[0], fraction=0.046, pad=0.02)

scb1 = axes_eb[1].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_bmode_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=snb_lo,
    vmax=snb_hi,
    linewidths=0,
    alpha=0.85,
)
axes_eb[1].set_xlabel("RA (deg)")
axes_eb[1].set_ylabel("Dec (deg)")
axes_eb[1].set_title(r"B-mode $\mathcal{S}/\mathcal{N}$")
axes_eb[1].invert_xaxis()
fig_eb.colorbar(scb1, ax=axes_eb[1], fraction=0.046, pad=0.02)
fig_eb.suptitle(
    f"Tract {CONFIG['TRACT']}, same filter/weights as E-mode row above",
    y=1.02,
)


In [ ]:
fig2, axp = plt.subplots(figsize=(7, 5.5))
sc = axp.scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-4,
    vmax=4,
    linewidths=0,
    alpha=0.85,
)
if peak_ipix.size:
    ra_peak, dec_peak = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    axp.scatter(
        ra_peak,
        dec_peak,
        s=120,
        facecolors="none",
        edgecolors="k",
        linewidths=1.2,
        label="peaks",
    )
    print(ra_peak, dec_peak)
axp.set_xlabel("RA (deg)")
axp.set_ylabel("Dec (deg)")
axp.scatter(149.7575000, 1.7666667, marker='*', color='k', label='[KLI2009] 246 - Galaxy Group')
axp.scatter(150.0430000, 2.5450000, marker='*', color='c', label='[FGH2007] 93 - Galaxy Cluster')
axp.scatter(149.7507083, 2.5198333, marker='*', color='g', label='[KLI2012] 964  - Galaxy Cluster')
axp.scatter(149.9167500, 2.4696389, marker='*', color='y', label='[KLI2012] 521  - Galaxy Cluster')


axp.set_title(r"E-mode $\mathcal{S}/\mathcal{N}$ with peak positions")
axp.invert_xaxis()
fig2.colorbar(sc, ax=axp, fraction=0.046, pad=0.02)
axp.legend(loc="upper left", framealpha=1)
fig2.tight_layout()


In [ ]:
if do_gnomonic:
    plt.figure(figsize=(7.0, 6.5))
    hp.gnomview(
        hp_sm,
        title=f"Tract {CONFIG['TRACT']} E-mode S/N (gnomonic)",
        rot=(rot_ra, rot_dec),
        xsize=600,
        reso=0.35,
        cmap="RdBu_r",
        min=-4,
        max=4,
        nest=False,
    )
    if peak_ipix.size:
        ra_pk, dec_pk = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
        hp.projscatter(
            ra_pk,
            dec_pk,
            lonlat=True,
            marker="o",
            s=80,
            facecolors="none",
            edgecolors="k",
            linewidths=1.0,
        )
    plt.show()


In [ ]:
if do_gnomonic:
    plt.figure(figsize=(7.0, 6.5))
    hp.gnomview(
        hp_smb,
        title=f"Tract {CONFIG['TRACT']} B-mode S/N (gnomonic)",
        rot=(rot_ra, rot_dec),
        xsize=600,
        reso=0.35,
        cmap="RdBu_r",
        min=-4,
        max=4,
        nest=False,
    )
    plt.show()


In [ ]:
# E-mode overlay on the binned tract gri color coadd.
# Follows DP2 deep_coadd access, DP1 103.6 PrettyPictureTask, DP0.2 03a make_lupton_rgb.
# Increase BIN_FACTOR if memory is tight. Set OVERLAY = "sn" for E-mode S/N.
# Tract color coadd (binned) + E-mode overlay inputs.
# BIN_FACTOR = 16 → ~1.66 deg tract at 0.2"/pix becomes ~1900 px on a side.
import gc

from astropy.visualization import make_lupton_rgb
from matplotlib.colors import TwoSlopeNorm

BIN_FACTOR = 16
RGB_BANDS = ("i", "r", "g")  # R, G, B as in DP1 103.6 / DP0.2 03a
COADD_DATASET_CANDIDATES = (
    "deep_coadd",  # DP2 CellCoadd
    "deep_coadd_cell_predetection",
    "deepCoadd",  # DP0.2 name
)
OVERLAY = "ap_mass"  # or "sn" for E-mode S/N
PRETTY_ABSMAX_NJY = 11000.0  # PrettyPictureTask white point (DP1 103.6)


def _query_refs(dataset_type, band):
    tract = int(CONFIG["TRACT"])
    try:
        return list(butler.query_datasets(dataset_type, tract=tract, band=band))
    except Exception:
        pass
    try:
        return list(
            butler.registry.queryDatasets(
                dataset_type,
                dataId={"tract": tract, "band": band, "skymap": CONFIG["SKYMAP"]},
            )
        )
    except Exception:
        return []


def _as_exposure(obj):
    """CellCoadd / MultipleCellCoadd → Exposure, else return as-is (DP2 stitch pattern)."""
    if obj is None:
        return None
    if hasattr(obj, "stitch"):
        stitched = obj.stitch()
        return stitched.asExposure() if hasattr(stitched, "asExposure") else stitched
    if hasattr(obj, "asExposure"):
        return obj.asExposure()
    return obj


def _get_attr(obj, *names, default=None):
    for name in names:
        if obj is None or not hasattr(obj, name):
            continue
        val = getattr(obj, name)
        return val() if callable(val) else val
    return default


def _image_array(exp):
    if hasattr(exp, "image"):
        return np.array(exp.image.array, dtype=np.float32, copy=True)
    if hasattr(exp, "array"):
        return np.array(exp.array, dtype=np.float32, copy=True)
    raise TypeError(f"Cannot read image array from {type(exp)!r}")


def _block_mean(arr, factor):
    """Mean-bin a 2D array; crop to a multiple of factor. NaNs are ignored."""
    ny, nx = arr.shape
    ny_b = (ny // factor) * factor
    nx_b = (nx // factor) * factor
    if ny_b < factor or nx_b < factor:
        return None, 0, 0
    blocks = np.asarray(arr[:ny_b, :nx_b], dtype=np.float64).reshape(
        ny_b // factor, factor, nx_b // factor, factor
    )
    with np.errstate(all="ignore"):
        out = np.nanmean(blocks, axis=(1, 3)).astype(np.float32)
    return out, ny_b, nx_b


def _intersect_bbox(a, b):
    if a is None or b is None:
        return None
    if hasattr(a, "clippedTo"):
        out = a.clippedTo(b)
        return None if out.isEmpty() else out
    x0 = max(a.getMinX(), b.getMinX())
    y0 = max(a.getMinY(), b.getMinY())
    x1 = min(a.getMaxX(), b.getMaxX())
    y1 = min(a.getMaxY(), b.getMaxY())
    if x1 < x0 or y1 < y0:
        return None
    from lsst.geom import Box2I, Point2I

    return Box2I(Point2I(x0, y0), Point2I(x1, y1))


try:
    skymap = butler.get("skyMap")
except Exception:
    skymap = butler.get("skyMap", dataId={"skymap": CONFIG["SKYMAP"]})
try:
    tract_info = skymap[int(CONFIG["TRACT"])]
except Exception:
    tract_info = skymap.generateTract(int(CONFIG["TRACT"]))
tract_wcs = _get_attr(tract_info, "wcs", "getWcs")
tract_bbox = _get_attr(tract_info, "outer_bbox", "bbox", "getOuterBBox", "getBBox")
if tract_wcs is None or tract_bbox is None:
    raise RuntimeError("Could not read tract WCS / bbox from skyMap")

coadd_dataset = None
patch_ids = []
for _dtype in COADD_DATASET_CANDIDATES:
    _refs = _query_refs(_dtype, RGB_BANDS[0])
    _patches = sorted({ref.dataId["patch"] for ref in _refs}, key=lambda p: str(p))
    if _patches:
        coadd_dataset = _dtype
        patch_ids = _patches
        break
if not patch_ids:
    raise RuntimeError(
        "No g/r/i coadd patches found. Tried: " + ", ".join(COADD_DATASET_CANDIDATES)
    )
print(f"Using {coadd_dataset} for tract {CONFIG['TRACT']}: {len(patch_ids)} patches")

nx_full = int(tract_bbox.getWidth())
ny_full = int(tract_bbox.getHeight())
nx = nx_full // BIN_FACTOR
ny = ny_full // BIN_FACTOR
mosaic = {band: np.full((ny, nx), np.nan, dtype=np.float32) for band in RGB_BANDS}
print(
    f"Binned mosaic {nx}×{ny} pixels "
    f"(BIN_FACTOR={BIN_FACTOR}; full tract {nx_full}×{ny_full})"
)

for patch_id in tqdm(patch_ids, desc="Coadd patches"):
    exps = {}
    try:
        for band in RGB_BANDS:
            obj = butler.get(
                coadd_dataset,
                dataId={"tract": int(CONFIG["TRACT"]), "patch": patch_id, "band": band},
            )
            exps[band] = _as_exposure(obj)
    except Exception as exc:
        print(f"  skip patch {patch_id}: {exc}")
        continue

    patch_info = None
    try:
        patch_info = tract_info[patch_id]
    except Exception:
        patch_info = None
    if patch_info is None and isinstance(patch_id, str) and "," in str(patch_id):
        try:
            ix, iy = (int(v) for v in str(patch_id).split(","))
            patch_info = tract_info[ix, iy]
        except Exception:
            patch_info = None
    if patch_info is None and hasattr(tract_info, "getPatchIndexPair"):
        try:
            patch_info = tract_info[tract_info.getPatchIndexPair(int(patch_id))]
        except Exception:
            patch_info = None
    inner = _get_attr(patch_info, "inner_bbox", "getInnerBBox") if patch_info is not None else None

    for band, exp in exps.items():
        arr = _image_array(exp)
        exp_bbox = _get_attr(exp, "getBBox")
        use_bbox = _intersect_bbox(exp_bbox, inner) if inner is not None else exp_bbox
        if use_bbox is None:
            use_bbox = exp_bbox
        if exp_bbox is not None and use_bbox is not None:
            x0 = use_bbox.getMinX() - exp_bbox.getMinX()
            y0 = use_bbox.getMinY() - exp_bbox.getMinY()
            arr = arr[y0 : y0 + use_bbox.getHeight(), x0 : x0 + use_bbox.getWidth()]
            origin_x, origin_y = use_bbox.getMinX(), use_bbox.getMinY()
        else:
            origin_x = tract_bbox.getMinX()
            origin_y = tract_bbox.getMinY()
        binned, ny_b, nx_b = _block_mean(arr, BIN_FACTOR)
        if binned is None:
            continue
        mx0 = (origin_x - tract_bbox.getMinX()) // BIN_FACTOR
        my0 = (origin_y - tract_bbox.getMinY()) // BIN_FACTOR
        my1 = min(my0 + binned.shape[0], ny)
        mx1 = min(mx0 + binned.shape[1], nx)
        if my0 >= ny or mx0 >= nx or my1 <= 0 or mx1 <= 0:
            continue
        sy0, sx0 = max(0, -my0), max(0, -mx0)
        mosaic[band][my0 + sy0 : my1, mx0 + sx0 : mx1] = binned[
            sy0 : my1 - my0, sx0 : mx1 - mx0
        ]
    del exps
    gc.collect()

rgb = None
try:
    from lsst.afw.image import ExposureF, ImageF
    from lsst.pipe.tasks.prettyPictureMaker import PrettyPictureTask
    from lsst.pipe.tasks.prettyPictureMaker._task import ChannelRGBConfig

    pretty_cfg = PrettyPictureTask.ConfigClass()
    if hasattr(pretty_cfg, "localContrastConfig"):
        pretty_cfg.localContrastConfig.doLocalContrast = False
    for _name in ("doPSFDeconcovlve", "doPSFDeconvolve"):
        if hasattr(pretty_cfg, _name):
            setattr(pretty_cfg, _name, False)
    if hasattr(pretty_cfg, "exposureBrackets"):
        pretty_cfg.exposureBrackets = None
    pretty_cfg.imageRemappingConfig.absMax = PRETTY_ABSMAX_NJY
    pretty_cfg.luminanceConfig.stretch = 750
    pretty_cfg.luminanceConfig.Q = 0.7
    pretty_cfg.luminanceConfig.highlight = 0.905882
    pretty_cfg.luminanceConfig.shadow = 0.15
    pretty_cfg.luminanceConfig.midtone = 0.3
    pretty_cfg.channelConfig["g"] = ChannelRGBConfig(r=0, g=0, b=1)
    pretty_cfg.channelConfig["r"] = ChannelRGBConfig(r=0, g=1, b=0)
    pretty_cfg.channelConfig["i"] = ChannelRGBConfig(r=1, g=0, b=0)
    pretty_task = PrettyPictureTask(config=pretty_cfg)

    def _arr_to_exp(arr):
        img = ImageF(arr.shape[1], arr.shape[0])
        img.array[:, :] = np.nan_to_num(arr, nan=0.0).astype(np.float32)
        exp = ExposureF(img.getBBox())
        exp.image.array[:, :] = img.array
        return exp

    rgb = pretty_task.run({b: _arr_to_exp(mosaic[b]) for b in RGB_BANDS}).outputRGB
    print("RGB from PrettyPictureTask (DP1 103.6)")
except Exception as exc:
    print(f"PrettyPictureTask unavailable ({exc}); using make_lupton_rgb (DP0.2 03a)")
    stretch = float(np.nanpercentile(np.abs(mosaic["i"]), 99))
    if not np.isfinite(stretch) or stretch <= 0:
        stretch = 80.0
    rgb = make_lupton_rgb(
        image_r=np.nan_to_num(mosaic["i"], nan=0.0),
        image_g=np.nan_to_num(mosaic["r"], nan=0.0),
        image_b=np.nan_to_num(mosaic["g"], nan=0.0),
        stretch=stretch,
        Q=8,
    )
    print(f"make_lupton_rgb stretch={stretch:.3g}")

print("RGB shape:", np.shape(rgb))

# Project HEALPix E-mode onto the binned mosaic and overlay as RdBu heatmap.
e_src = sn_ratio_map if OVERLAY == "sn" else ap_mass_map
e_label = r"E-mode $\mathcal{S}/\mathcal{N}$" if OVERLAY == "sn" else r"E-mode $M_{\rm ap}$"

e_full = np.full(hp.nside2npix(nside), np.nan, dtype=np.float64)
e_full[np.asarray(ipix_eval, dtype=np.int64)] = np.asarray(e_src, dtype=np.float64)

yy, xx = np.indices((ny, nx), dtype=np.float64)
x_tract = float(tract_bbox.getMinX()) + (xx + 0.5) * BIN_FACTOR
y_tract = float(tract_bbox.getMinY()) + (yy + 0.5) * BIN_FACTOR
ra_grid, dec_grid = tract_wcs.pixelToSkyArray(
    x_tract.ravel(), y_tract.ravel(), degrees=True
)
e_grid = e_full[
    hp.ang2pix(nside, ra_grid, dec_grid, nest=False, lonlat=True)
].reshape(ny, nx)
e_grid[~np.isfinite(mosaic["i"])] = np.nan

finite = np.isfinite(e_grid)
if not finite.any():
    raise RuntimeError("No finite E-mode pixels overlap the binned coadd")
vmax = float(np.nanpercentile(np.abs(e_grid[finite]), 95))
if not np.isfinite(vmax) or vmax <= 0:
    vmax = 1.0
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

rgba = plt.cm.RdBu_r(norm(np.nan_to_num(e_grid, nan=0.0)))
alpha = np.clip(np.abs(e_grid) / vmax, 0.0, 1.0) * 0.65
alpha[~finite] = 0.0
rgba[..., 3] = alpha

fig_ov, ax_ov = plt.subplots(figsize=(9.5, 8.5), constrained_layout=True)
ax_ov.imshow(rgb, origin="lower", interpolation="nearest")
ax_ov.imshow(rgba, origin="lower", interpolation="nearest")
sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
sm.set_array([])
cbar = fig_ov.colorbar(sm, ax=ax_ov, fraction=0.046, pad=0.02)
cbar.set_label(e_label)

if "peak_ipix" in globals() and np.size(peak_ipix):
    ra_pk, dec_pk = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    ra_pk = np.asarray(ra_pk, dtype=np.float64)
    dec_pk = np.asarray(dec_pk, dtype=np.float64)
    try:
        x_pk, y_pk = tract_wcs.skyToPixelArray(ra_pk, dec_pk, degrees=True)
    except Exception:
        from lsst.geom import SpherePoint, degrees as lsst_degrees

        xy_pk = [
            tract_wcs.skyToPixel(SpherePoint(float(ra), float(dec), lsst_degrees))
            for ra, dec in zip(ra_pk, dec_pk)
        ]
        x_pk = np.array([p.getX() for p in xy_pk])
        y_pk = np.array([p.getY() for p in xy_pk])
    ax_ov.scatter(
        (x_pk - tract_bbox.getMinX()) / BIN_FACTOR,
        (y_pk - tract_bbox.getMinY()) / BIN_FACTOR,
        s=90,
        facecolors="none",
        edgecolors="yellow",
        linewidths=1.2,
        label="E-mode peaks",
    )
    ax_ov.legend(loc="upper right", framealpha=0.9)

nxt = np.linspace(0, max(nx - 1, 0), 5)
nyt = np.linspace(0, max(ny - 1, 0), 5)
x_tick = float(tract_bbox.getMinX()) + (nxt + 0.5) * BIN_FACTOR
y_tick = float(tract_bbox.getMinY()) + (nyt + 0.5) * BIN_FACTOR
y_mid = float(tract_bbox.getMinY()) + 0.5 * ny * BIN_FACTOR
x_mid = float(tract_bbox.getMinX()) + 0.5 * nx * BIN_FACTOR
ra_tick, _ = tract_wcs.pixelToSkyArray(x_tick, np.full_like(x_tick, y_mid), degrees=True)
_, dec_tick = tract_wcs.pixelToSkyArray(np.full_like(y_tick, x_mid), y_tick, degrees=True)
ax_ov.set_xticks(nxt)
ax_ov.set_xticklabels([f"{ra:.2f}°" for ra in ra_tick])
ax_ov.set_yticks(nyt)
ax_ov.set_yticklabels([f"{dec:.2f}°" for dec in dec_tick])
ax_ov.set_title(
    f"Tract {CONFIG['TRACT']} {''.join(RGB_BANDS)} coadd + {e_label} "
    f"(bin×{BIN_FACTOR}, {coadd_dataset})"
)
ax_ov.set_xlabel("RA")
ax_ov.set_ylabel("Dec")
ax_ov.set_aspect("equal")
plt.show()
